# Empirical Bias-Variance Decomposition Simulation

This notebook demonstrates the mathematical **Bias-Variance Decomposition** using an empirical Monte Carlo simulation. We define a true target function, generate multiple independent training splits, train estimators of varying complexity (polynomial regression models), and calculate their empirical bias, variance, and expected test error.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

np.random.seed(42)

# 1. Define the true function f(x) and simulation parameters
def true_f(x):
    return np.sin(np.pi * x)

sigma = 0.5  # Irreducible noise standard deviation (sigma^2 = 0.25)
m = 20       # Size of each training dataset
N = 100      # Number of datasets to simulate (realizations)

# Generate fixed test points to evaluate expectation
x_test = np.linspace(-1, 1, 100)
y_test_true = true_f(x_test)

print(f"Noise variance (sigma^2): {sigma**2:.4f}")

## 2. Simulate N = 100 Training Datasets and Fit Models

We generate 100 different datasets. For each dataset, we fit three models of different complexity:
- **Degree 1 (Linear):** High Bias, Low Variance.
- **Degree 3 (Cubic):** Well-balanced.
- **Degree 12 (Overparameterized):** Low Bias, High Variance.

In [ ]:
degrees = [1, 3, 12]
predictions = {d: [] for d in degrees}

for _ in range(N):
    # Generate random training dataset
    x_train = np.random.uniform(-1, 1, m)
    y_train = true_f(x_train) + np.random.normal(0, sigma, m)
    
    # Fit models
    for d in degrees:
        model = make_pipeline(PolynomialFeatures(d), LinearRegression())
        model.fit(x_train.reshape(-1, 1), y_train)
        # Predict on test points
        preds = model.predict(x_test.reshape(-1, 1))
        predictions[d].append(preds)

# Convert to numpy arrays for calculation (shape: N_datasets, test_points)
for d in degrees:
    predictions[d] = np.array(predictions[d])
print("Simulation completed.")

## 3. Calculate Empirical Bias, Variance, and Test Error

For each model class, we compute:
1. **Expected Prediction:** $E[\hat{f}(x_0)] = \frac{1}{N} \sum_{k=1}^N \hat{f}_{D_k}(x_0)$
2. **Bias squared:** $[\text{Bias}(\hat{f}(x_0))]^2 = (E[\hat{f}(x_0)] - f(x_0))^2$
3. **Variance:** $\text{Var}(\hat{f}(x_0)) = \frac{1}{N} \sum_{k=1}^N (\hat{f}_{D_k}(x_0) - E[\hat{f}(x_0)])^2$
4. **Expected Test MSE:** $E[(y_0 - \hat{f}(x_0))^2]$ (averaged across all splits)

In [ ]:
for d in degrees:
    preds_matrix = predictions[d]
    
    # Compute expectations over N splits
    expected_preds = np.mean(preds_matrix, axis=0)
    
    # Bias squared
    bias_sq = (expected_preds - y_test_true) ** 2
    
    # Variance
    variance = np.mean((preds_matrix - expected_preds) ** 2, axis=0)
    
    # Test MSE (true target includes noise)
    test_mse = np.mean((preds_matrix - (y_test_true + np.random.normal(0, sigma, size=y_test_true.shape))) ** 2, axis=0)
    
    # Average across all test points
    avg_bias_sq = np.mean(bias_sq)
    avg_variance = np.mean(variance)
    avg_test_mse = np.mean(test_mse)
    
    print(f"\nDegree {d:2d} | Avg Bias^2: {avg_bias_sq:.4f} | Avg Var: {avg_variance:.4f} | Sum: {avg_bias_sq + avg_variance + sigma**2:.4f} | Test MSE: {avg_test_mse:.4f}")

### Verification
Notice how the **Sum** (Bias$^2$ + Variance + $\sigma^2$) matches the **Test MSE** almost perfectly (any tiny deviation is due to using a finite $N=100$ simulation instead of infinite splits).

## 4. Visualize the Fits

Let's plot the individual predictions for the Degree 1 and Degree 12 models. Notice how the linear model is stable but fits poorly (unbiased mean is a straight line), while the degree 12 model fluctuates wildly across different splits (high variance).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot Degree 1
for i in range(15):  # plot 15 realizations
    axes[0].plot(x_test, predictions[1][i], color='#ff922b', alpha=0.3)
axes[0].plot(x_test, np.mean(predictions[1], axis=0), color='#e03131', linewidth=2.5, label='E[f_hat] (Average model)')
axes[0].plot(x_test, y_test_true, color='black', linestyle='--', label='f(x) (True function)')
axes[0].set_title("Degree 1: High Bias, Low Variance")
axes[0].legend()
axes[0].grid(True)

# Plot Degree 12
for i in range(15):
    axes[1].plot(x_test, predictions[12][i], color='#339af0', alpha=0.3)
axes[1].plot(x_test, np.mean(predictions[12], axis=0), color='#1971c2', linewidth=2.5, label='E[f_hat] (Average model)')
axes[1].plot(x_test, y_test_true, color='black', linestyle='--', label='f(x) (True function)')
axes[1].set_ylim(-2, 2)  # bound due to extreme variance swings
axes[1].set_title("Degree 12: Low Bias, High Variance")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()